## this notebook is for reading dataset and store it into a csv format with pandas or pyspark.

In [0]:
import os
path = "/Volumes/workspace/fall_detection_project/raw_data/"
print(os.listdir(path))

In [0]:
import sys
import os

# This tells Python where to find your src/ folder
sys.path.insert(0, os.path.abspath(".."))

from src.config import CFG

# Quick sanity check — print key config values
print("Config loaded successfully ✅")
print(f"Dataset path  : {CFG['data']['zip_path']}")
print(f"Sample rate   : {CFG['data']['sample_rate']} Hz")
print(f"Train subjects: {len(CFG['data']['train_subjects'])}")
print(f"Test subjects : {len(CFG['data']['test_subjects'])}")

In [0]:

zip_path = CFG["data"]["zip_path"]

if os.path.exists(zip_path):
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"File found ✅")
    print(f"Path : {zip_path}")
    print(f"Size : {size_mb:.1f} MB")
else:
    print(f"File NOT found ❌")
    print(f"Looked at: {zip_path}")

In [0]:
# Verify config has the right value
print(f"base_path_in_zip: {CFG['data']['base_path_in_zip']}")

# Manually check if one subject's files are found
import zipfile

with zipfile.ZipFile(CFG["data"]["zip_path"], "r") as zf:
    sa01_files = [
        f for f in zf.namelist()
        if f.startswith("DataSet/SisFall_dataset/SA01/")
        and f.endswith(".txt")
        and not f.startswith("__MACOSX")
    ]
    print(f"\nFiles found for SA01: {len(sa01_files)}")
    print("First 5:")
    for f in sa01_files[:5]:
        print(f)

In [0]:
from src.functions import read_zip

# Read only train subjects for now
all_data, all_labels, activity_codes, file_names = read_zip(
    zip_path         = CFG["data"]["zip_path"],
    base_path_in_zip = CFG["data"]["base_path_in_zip"],
    subject_ids      = CFG["data"]["train_subjects"],
)

In [0]:
import numpy as np

# How many signals do we have?
print(f"Number of signals : {len(all_data)}")
print(f"Number of labels  : {len(all_labels)}")

# Look at the first signal
first_signal = all_data[0]
print(f"\nFirst signal:")
print(f"  Type   : {type(first_signal)}")
print(f"  Shape  : {first_signal.shape}")
print(f"  Label  : {all_labels[0]}")
print(f"  File   : {file_names[0]}")

# What are the unique labels?
unique, counts = np.unique(all_labels, return_counts=True)
print(f"\nClass distribution:")
for label, count in zip(unique, counts):
    print(f"  {label} : {count}")

In [0]:
import matplotlib.pyplot as plt

lengths = [s.shape[1] for s in all_data]

print(f"Min length  : {min(lengths)} samples ({min(lengths)/200:.1f} sec)")
print(f"Max length  : {max(lengths)} samples ({max(lengths)/200:.1f} sec)")
print(f"Mean length : {int(sum(lengths)/len(lengths))} samples ({sum(lengths)/len(lengths)/200:.1f} sec)")

# Separate by label
adl_lengths  = [all_data[i].shape[1] for i in range(len(all_data)) if all_labels[i] == "ADL"]
fall_lengths = [all_data[i].shape[1] for i in range(len(all_data)) if all_labels[i] == "Fall"]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(adl_lengths,  bins=40, color="steelblue", edgecolor="white")
axes[0].set_title("ADL — Signal Length Distribution")
axes[0].set_xlabel("Samples")
axes[0].set_ylabel("Count")
axes[0].grid()

axes[1].hist(fall_lengths, bins=40, color="tomato", edgecolor="white")
axes[1].set_title("Fall — Signal Length Distribution")
axes[1].set_xlabel("Samples")
axes[1].set_ylabel("Count")
axes[1].grid()

plt.tight_layout()
plt.show()

In [0]:
# Find one ADL and one Fall signal
adl_idx  = all_labels.index("ADL")
fall_idx = all_labels.index("Fall")

adl_signal  = all_data[adl_idx]
fall_signal = all_data[fall_idx]

fig, axes = plt.subplots(2, 2, figsize=(16, 8))

# ADL — accelerometer
time_adl = range(adl_signal.shape[1])
axes[0][0].plot(time_adl, adl_signal[0], label="Acc X", alpha=0.8)
axes[0][0].plot(time_adl, adl_signal[1], label="Acc Y", alpha=0.8)
axes[0][0].plot(time_adl, adl_signal[2], label="Acc Z", alpha=0.8)
axes[0][0].set_title(f"ADL — Accelerometer ({file_names[adl_idx]})")
axes[0][0].set_xlabel("Samples")
axes[0][0].set_ylabel("Value")
axes[0][0].legend()
axes[0][0].grid()

# ADL — gyroscope
axes[0][1].plot(time_adl, adl_signal[3], label="Gyro X", alpha=0.8)
axes[0][1].plot(time_adl, adl_signal[4], label="Gyro Y", alpha=0.8)
axes[0][1].plot(time_adl, adl_signal[5], label="Gyro Z", alpha=0.8)
axes[0][1].set_title(f"ADL — Gyroscope ({file_names[adl_idx]})")
axes[0][1].set_xlabel("Samples")
axes[0][1].set_ylabel("Value")
axes[0][1].legend()
axes[0][1].grid()

# Fall — accelerometer
time_fall = range(fall_signal.shape[1])
axes[1][0].plot(time_fall, fall_signal[0], label="Acc X", alpha=0.8, color="tomato")
axes[1][0].plot(time_fall, fall_signal[1], label="Acc Y", alpha=0.8, color="coral")
axes[1][0].plot(time_fall, fall_signal[2], label="Acc Z", alpha=0.8, color="red")
axes[1][0].set_title(f"Fall — Accelerometer ({file_names[fall_idx]})")
axes[1][0].set_xlabel("Samples")
axes[1][0].set_ylabel("Value")
axes[1][0].legend()
axes[1][0].grid()

# Fall — gyroscope
axes[1][1].plot(time_fall, fall_signal[3], label="Gyro X", alpha=0.8, color="tomato")
axes[1][1].plot(time_fall, fall_signal[4], label="Gyro Y", alpha=0.8, color="coral")
axes[1][1].plot(time_fall, fall_signal[5], label="Gyro Z", alpha=0.8, color="red")
axes[1][1].set_title(f"Fall — Gyroscope ({file_names[fall_idx]})")
axes[1][1].set_xlabel("Samples")
axes[1][1].set_ylabel("Value")
axes[1][1].legend()
axes[1][1].grid()

plt.suptitle("Sample Signals — ADL vs Fall", fontsize=14)
plt.tight_layout()
plt.show()

In [0]:
import pandas as pd
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Build a summary dataframe of what we loaded
meta_df = pd.DataFrame({
    "file_name"     : file_names,
    "label"         : all_labels,
    "activity_code" : activity_codes,
    "n_samples"     : [s.shape[1] for s in all_data],
    "duration_sec"  : [round(s.shape[1] / CFG["data"]["sample_rate"], 2) for s in all_data],
    "n_axes"        : [s.shape[0] for s in all_data],
})

# Convert to Spark DataFrame and save as Delta table
spark_df = spark.createDataFrame(meta_df)
spark_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.fall_detection_project.bronze_metadata")

print("Delta table saved ✅")
print(f"Rows : {meta_df.shape[0]}")
display(spark_df.limit(10))

In [0]:
print(f"Train: {len(CFG['data']['train_subjects'])}")
print(f"Test : {len(CFG['data']['test_subjects'])}")